Configuration

In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import precision_recall_curve, auc
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from xgboost import XGBClassifier

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# 1. Generación de datos sintéticos (1% fraude)
X, y = make_classification(
    n_samples=10000, 
    n_features=5, 
    n_informative=4, 
    n_redundant=1, 
    weights=[0.99, 0.01], 
    random_state=42
)

# 2. Creación del DataFrame base
columns_base = [
    'Monto_Transaccion_Normalizado', 
    'Distancia_Ubicacion_Km', 
    'Frecuencia_Compras_Ultima_Hora', 
    'Edad_Cuenta_Dias', 
    'Dispositivo_Sospechoso_Score'
]
df_transacciones = pd.DataFrame(X, columns=columns_base)
df_transacciones['Es_Fraude'] = y

# Visualizamos las primeras 5 filas
print("Primeras transacciones registradas:")
display(df_transacciones.head())

conteo_fraudes = df_transacciones['Es_Fraude'].value_counts()
print("\nDistribución de clases (0 = Legítima, 1 = Fraude):")
print(conteo_fraudes)
print(f"\nPorcentaje de fraude en el dataset: {(conteo_fraudes[1] / len(df_transacciones)) * 100:.2f}%")

Primeras transacciones registradas:


,Monto_Transaccion_Normalizado,Distancia_Ubicacion_Km,Frecuencia_Compras_Ultima_Hora,Edad_Cuenta_Dias,Dispositivo_Sospechoso_Score,Es_Fraude
0,-2.299933,3.242833,1.879708,0.066115,1.102833,0
1,-1.707593,0.122327,-0.604671,-0.905535,-0.692637,0
2,-0.916655,-0.569816,-0.719821,-0.782531,-0.772442,0
3,0.805849,-1.691910,-2.878510,-0.472700,-0.355966,0
4,-2.202056,2.670985,-2.196414,0.248894,0.691424,0



Distribución de clases (0 = Legítima, 1 = Fraude):
Es_Fraude
0    9844
1     156
Name: count, dtype: int64

Porcentaje de fraude en el dataset: 1.56%


In [52]:
X_features = df_transacciones[columns_base]
y_target = df_transacciones['Es_Fraude']

# --- Modelo 1: Isolation Forest (No supervisado) ---
iso_forest = IsolationForest(contamination=0.0156, random_state=42)
pred_iso = iso_forest.fit_predict(X_features)
pred_iso_binary = [1 if p == -1 else 0 for p in pred_iso]

print("--- ISOLATION FOREST ---")
print(classification_report(y_target, pred_iso_binary))

# --- Split para modelos supervisados (80/20 Stratified) ---
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.2, random_state=42, stratify=y_target
)

# --- Modelo 2: Random Forest (Supervisado) ---
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# Evaluación con ajuste de umbral (0.2)
rf_probs = rf_model.predict_proba(X_test)[:, 1]
rf_preds = (rf_probs >= 0.2).astype(int)

print("--- RANDOM FOREST (Umbral = 0.2) ---")
print(classification_report(y_test, rf_preds))

--- ISOLATION FOREST ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      9844
           1       0.07      0.07      0.07       156

    accuracy                           0.97     10000
   macro avg       0.53      0.53      0.53     10000
weighted avg       0.97      0.97      0.97     10000

--- RANDOM FOREST (Umbral = 0.2) ---
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1969
           1       0.77      0.55      0.64        31

    accuracy                           0.99      2000
   macro avg       0.88      0.77      0.82      2000
weighted avg       0.99      0.99      0.99      2000



In [53]:
# Creación de variables de negocio sobre el DataFrame completo
df_fe = df_transacciones.copy()

df_fe['Riesgo_Geo_Dispositivo'] = df_fe['Distancia_Ubicacion_Km'] * df_fe['Dispositivo_Sospechoso_Score']
df_fe['Intensidad_Compra'] = df_fe['Frecuencia_Compras_Ultima_Hora'] / (df_fe['Edad_Cuenta_Dias'].abs() + 1)

# Nuevo Split con las nuevas variables
X_fe = df_fe.drop(columns=['Es_Fraude'])
y_fe = df_fe['Es_Fraude']

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe, y_fe, test_size=0.2, random_state=42, stratify=y_fe
)

In [54]:
# Cálculo del factor de desbalance
scale_pos_weight_val = (y_train_fe == 0).sum() / (y_train_fe == 1).sum()

# Espacio de Búsqueda de Hiperparámetros
param_grid = {
    'n_estimators': [100, 150, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'scale_pos_weight': [int(scale_pos_weight_val), int(scale_pos_weight_val * 0.5), int(scale_pos_weight_val * 1.5)]
}

# Búsqueda Aleatoria optimizando PR-AUC (average_precision)
random_search = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=param_grid,
    n_iter=25,
    scoring='average_precision',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_fe, y_train_fe)
best_xgb = random_search.best_estimator_

# Evaluación Final
y_probs_xgb = best_xgb.predict_proba(X_test_fe)[:, 1]
y_pred_xgb = (y_probs_xgb >= 0.5).astype(int)

precision, recall, _ = precision_recall_curve(y_test_fe, y_probs_xgb)
pr_auc_final = auc(recall, precision)

print("\n--- MODELO OPTIMIZADO (XGBoost Final) ---")
print(f"Mejores Parámetros: {random_search.best_params_}")
print(f"PR-AUC Final: {pr_auc_final:.4f}")
print("\nReporte de Clasificación:")
print(classification_report(y_test_fe, y_pred_xgb))

Fitting 3 folds for each of 25 candidates, totalling 75 fits

--- MODELO OPTIMIZADO (XGBoost Final) ---
Mejores Parámetros: {'subsample': 0.8, 'scale_pos_weight': 31, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
PR-AUC Final: 0.5305

Reporte de Clasificación:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1969
           1       0.57      0.55      0.56        31

    accuracy                           0.99      2000
   macro avg       0.78      0.77      0.78      2000
weighted avg       0.99      0.99      0.99      2000



In [55]:
# 1. Obtenemos las probabilidades predichas para la clase positiva (fraude = 1) en el conjunto de prueba
# [:, 1] nos da la probabilidad exacta de que sea fraude según Random Forest
y_probs = rf_model.predict_proba(X_test)[:, 1]

# 2. EXPERIMENTO DE UMBRAL (Threshold Tuning)
# Por defecto es 0.5. Vamos a bajarlo a 0.2 para ser más estrictos y atrapar más fraudes.
nuevo_umbral = 0.2
y_pred_nuevo_umbral = (y_probs >= nuevo_umbral).astype(int)

print(f"--- Matriz de Confusión con Umbral Ajustado ({nuevo_umbral}) ---")
print(confusion_matrix(y_test, y_pred_nuevo_umbral))
print(f"\n--- Reporte de Clasificación (Umbral = {nuevo_umbral}) ---")
print(classification_report(y_test, y_pred_nuevo_umbral))

# 3. GRAFICO DE CURVA PRECISION-RECALL (PR-AUC)
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)
pr_auc = auc(recall, precision)

print(f"Área bajo la curva Precision-Recall (PR-AUC): {pr_auc:.4f}")

--- Matriz de Confusión con Umbral Ajustado (0.2) ---
[[1964    5]
 [  14   17]]

--- Reporte de Clasificación (Umbral = 0.2) ---
              precision    recall  f1-score   support

           0       0.99      1.00      1.00      1969
           1       0.77      0.55      0.64        31

    accuracy                           0.99      2000
   macro avg       0.88      0.77      0.82      2000
weighted avg       0.99      0.99      0.99      2000

Área bajo la curva Precision-Recall (PR-AUC): 0.5904
